In [1]:
from answering import *
from indexing import *

c:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedder  = OpenAIEmbedder(model="text-embedding-3-small")
store     = ChromaRAGStore(dir="./chroma_db", collection_name="crack_detection")
reranker  = CrossEncoderReranker(model_name=DEFAULT_RERANKER_MODEL)
generator = AnswerGenerator(model=DEFAULT_GENERATION_MODEL)

# Engineering maintenance report
payload = InspectionPayload.from_dict({
    "image_id":                  "bridge_01.jpg",
    "crack_detected":            True,
    "crack_area_ratio":          0.054,
    "estimated_crack_length_px": 438,
    "num_crack_regions":         3,
    "severity":                  "High",
    "model_confidence":          0.91,
})

chroma_results = query_by_inspection(
    payload=payload,
    embedder=embedder,
    store=store,
    top_k=DEFAULT_RETRIEVAL_TOP_K,
)

report = generate_maintenance_report(
    payload=payload,
    embedder=embedder,
    store=store,
    chroma_results=chroma_results,
    reranker=reranker,
    generator=generator,
)
print_maintenance_report(report)

Search returned 20 result(s).
Search returned 5 result(s).

  STRUCTURAL INSPECTION MAINTENANCE REPORT
  Asset / Image  : bridge_01.jpg
  Severity       : High
  Generated At   : 2026-03-24T03:38:03.927201+00:00

──────────────────────────────────────────────────────────────
  1. FINDINGS SUMMARY
──────────────────────────────────────────────────────────────
  The inspection of bridge_01.jpg identified crack presence as True, with a crack area ratio of 0.0540 and an estimated crack length of 438 px. The crack was detected in 3 distinct crack regions. The reported severity was High, and the model confidence was 0.91.

──────────────────────────────────────────────────────────────
  2. RISK ASSESSMENT
──────────────────────────────────────────────────────────────
  Cracking of structural significance may indicate overstress; however, most forms of concrete deterioration are generally most significant in their effects on durability rather than strength. [3] Given the reported High severit

In [7]:
context_texts = [source.chunk_text for source in report.sources]
context = " ".join(context_texts)
print(context)

When reporting cracks record the length, width, location, and orientation (horizontal, vertical, diagonal, etc.) of each crack. Also indicate the presence of rust stains, efflorescence, or evidence of differential movement on either side of the crack. Cracks may also be an indication of overloading, corrosion of the reinforcing steel, or settlement of the structure. Report each crack in terms of; the length, width, location, and orientation (horizontal, vertical, diagonal, etc.). Also, indicate the presence of rust stains, efflorescence, or evidence of differential movement on either side of the crack. While cracking of structural significance may indicate overstress, most forms of concrete deterioration (spalling, scaling, efflorescence) usually are most significant in their effects on durability rather than strength. Once the cause of cracking has been established beyond doubt, and any possible steps have been taken to avoid further movement, it is possible to restore the structure t

In [6]:
section_name = "risk_assessment"
section_text = getattr(report, section_name, "")
print(section_text)

Cracking of structural significance may indicate overstress; however, most forms of concrete deterioration are generally most significant in their effects on durability rather than strength. [3] Given the reported High severity, the condition should be treated as potentially significant and further crack characterization should be undertaken to support assessment and maintenance decisions. [1] The inspection reporting should include crack length, width, location, and orientation (horizontal, vertical, diagonal, etc.), and should also note whether rust stains, efflorescence, or evidence of differential movement are present on either side of the crack. [1] Cracks may also be an indication of overloading, corrosion of the reinforcing steel, or settlement of the structure, and these potential causes should be considered during follow-up evaluation. [2]


In [8]:
section_name = "repair_actions"
section_text = getattr(report, section_name, "")
print(section_text)

1. Each crack should be reported with its length, width, location, and orientation, and the presence of rust stains, efflorescence, or evidence of differential movement on either side of the crack should be documented to support cause identification and selection of treatment. [1]  
2. Once the cause of cracking has been established beyond doubt, and any possible steps have been taken to avoid further movement, the structure may be restored to its original strength and durability by injecting the cracks full depth with epoxy resin specifically developed for such an application. [4]  
3. If the crack width is more than 0.1 mm and the surfaces of the concrete in the crack are clean and sound, cracks can be successfully filled and repaired by specialised controlled pressure-injection techniques. [4]


In [9]:
class LLMGroundingEvaluator:
    """
    LLM-based evaluation of section grounding in retrieved context.
    Uses LLM to assess semantic coherence between report output and context.
    """
    
    def __init__(self, model: str = "gpt-5.4-nano"):
        self.client = OpenAI()
        self.model = model
    
    def evaluate_section(
        self,
        section_text: str,
        section_name: str,
        context_texts: List[str],
    ) -> dict:
        """
        Use LLM to evaluate if section is grounded in context.
        
        Args:
            section_text (str): Text of that particular section.
            section_name (str): Name of that particular section.  
            context_texts (List[str]): retrieved text from report.sources
        
        Returns:
            {
                "is_grounded": bool,
                "confidence": float (0.0-1.0),
                "reasoning": str,
                "hallucination_risk": "low" | "medium" | "high"
            }
        """
        context_joined = " ".join(context_texts)
        
        prompt = f"""Evaluate if this {section_name} is grounded in the provided context.

GENERATED SECTION:
{section_text}

RETRIEVED CONTEXT:
{context_joined}

Assess:
1. Are the main claims supported by the context?
2. Does the section contain information NOT in the context (hallucination)?
3. Overall grounding quality

Respond with JSON:
{{
    "is_grounded": true/false,
    "confidence": 0.0-1.0,
    "hallucination_risk": "low|medium|high",
    "reasoning": "brief explanation"
}}"""
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        
        result = json.loads(response.choices[0].message.content)
        return result
    
    def evaluate_report(
        self,
        report: MaintenanceReport,
        context_texts: List[str],
    ) -> dict:
        """Evaluate all sections and return aggregated results."""
        
        section_fields = [
            ("risk_assessment", "Risk Assessment"),
            ("repair_actions", "Repair Actions"),
            ("inspection_schedule", "Inspection Schedule"),
        ]
        
        results = {}
        for field_name, label in section_fields:
            section_text = getattr(report, field_name, "")
            if section_text:
                results[field_name] = self.evaluate_section(
                    section_text, label, context_texts
                )
        
        # Aggregate
        hallucination_risks = [
            r.get("hallucination_risk", "high") 
            for r in results.values()
        ]
        
        overall_risk = (
            "high" if "high" in hallucination_risks
            else "medium" if "medium" in hallucination_risks
            else "low"
        )
        
        return {
            "sections": results,
            "overall_hallucination_risk": overall_risk,
        }

In [10]:
evaluator = LLMGroundingEvaluator()
llm_results = evaluator.evaluate_report(report, context_texts)

print(f"Risk: {llm_results['overall_hallucination_risk']}")
for section, result in llm_results['sections'].items():
    print(f"  {section}: {result['hallucination_risk']} - {result['reasoning']}")

Risk: low
  risk_assessment: low - The section’s key claims are largely supported by the retrieved context: it advises recording crack length/width/location/orientation and noting rust stains/efflorescence/differential movement; it also states cracks can indicate overloading, reinforcing steel corrosion, or settlement. The statement that cracking of structural significance may indicate overstress and that concrete deterioration is often more significant for durability than strength is also consistent with the context. Minor potential overreach: it recommends further crack characterization based on 'reported High severity' and mentions maintenance decisions, which are not explicitly detailed in the context, but this is a reasonable extension rather than a clear contradiction. No clear unsupported technical claims (e.g., epoxy injection details) are introduced.
  repair_actions: low - All three repair-action claims closely match the retrieved context: documenting crack length/width/locat